In [ ]:
# Install dependencies
!pip install -q faster-whisper
!apt-get -qq install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 40.7 MB/s eta 0:00:00


In [ ]:
# Unzip audio files
!unzip AUD.zip -d /content/audio_data

Archive:  AUD.zip
   creating: /content/audio_data/AUD/
  inflating: /content/audio_data/AUD/MarauliKhurad1.m4a  
  inflating: /content/audio_data/AUD/AUD-20260402-WA0018.m4a  
  inflating: /content/audio_data/AUD/MarauliKhurad2.m4a  
  inflating: /content/audio_data/AUD/MarauliKhurad3.m4a  
  inflating: /content/audio_data/AUD/Full Narration_MarauliKhurad.m4a  


In [ ]:
import os
from faster_whisper import WhisperModel

# Load model
model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

audio_folder = "/content/audio_data/AUD/"
output_folder = "/content/transcripts/"
os.makedirs(output_folder, exist_ok=True)

audio_extensions = (".mp3", ".wav", ".m4a", ".ogg", ".flac")

files = [
    f for f in os.listdir(audio_folder)
    if f.lower().endswith(audio_extensions)
]

print(f"Found {len(files)} audio files")

for filename in files:

    audio_path = os.path.join(audio_folder, filename)

    print(f"\nProcessing: {filename}")

    segments, info = model.transcribe(
        audio_path,
        language="pa",
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=True
    )

    transcript = ""
    timestamped = ""

    for seg in segments:

        transcript += seg.text.strip() + " "

        timestamped += (
            f"[{seg.start:.2f} --> {seg.end:.2f}] "
            f"{seg.text.strip()}\n"
        )

    base_name = os.path.splitext(filename)[0]

    transcript_file = os.path.join(
        output_folder,
        f"{base_name}.txt"
    )

    timestamp_file = os.path.join(
        output_folder,
        f"{base_name}_timestamps.txt"
    )

    with open(transcript_file, "w", encoding="utf-8") as f:
        f.write(transcript.strip())

    with open(timestamp_file, "w", encoding="utf-8") as f:
        f.write(timestamped)

    print(f"Saved: {transcript_file}")
    print(f"Saved: {timestamp_file}")

print("\nAll files processed.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Found 5 audio files

Processing: AUD-20260402-WA0018.m4a
Saved: /content/transcripts/AUD-20260402-WA0018.txt
Saved: /content/transcripts/AUD-20260402-WA0018_timestamps.txt

Processing: MarauliKhurad2.m4a
Saved: /content/transcripts/MarauliKhurad2.txt
Saved: /content/transcripts/MarauliKhurad2_timestamps.txt

Processing: MarauliKhurad3.m4a
Saved: /content/transcripts/MarauliKhurad3.txt
Saved: /content/transcripts/MarauliKhurad3_timestamps.txt

Processing: MarauliKhurad1.m4a
Saved: /content/transcripts/MarauliKhurad1.txt
Saved: /content/transcripts/MarauliKhurad1_timestamps.txt

Processing: Full Narration_MarauliKhurad.m4a
Saved: /content/transcripts/Full Narration_MarauliKhurad.txt
Saved: /content/transcripts/Full Narration_MarauliKhurad_timestamps.txt

All files processed.
